# 1 — Esplorare i file grezzi

A cosa serve: aprire un CSV qualsiasi di `data/raw/`, vedere com'è fatto e
guardarci dentro, senza ricordarsi percorsi e nomi di file.

Se un file non c'è, scaricalo:

```
python3 -m lib.download tml --tour atp
python3 -m lib.download mcp --gender m
```

In [ ]:
# Setup: rende importabile lib/ da qualunque cartella si lanci il notebook
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "lib").is_dir():          # risale finché non trova la radice del progetto
    if ROOT.parent == ROOT:                 # arrivati a "/": meglio un errore chiaro
        raise RuntimeError("lanciare il notebook da dentro tennis_project/")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from lib import explore, loaders, catalog

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)

print("radice:", ROOT)

## Quali file ho

Righe, colonne e dimensione di tutto ciò che è stato scaricato.

In [ ]:
explore.listing()

In [ ]:
# Filtra per nome quando i file sono tanti
explore.listing("stats")

## Aprire un file

`explore.read()` accetta un pezzo del nome: non serve il percorso completo.
Cambia `FILE` e riesegui da qui in giù.

In [ ]:
FILE = "charting-m-stats-Rally.csv"

df = explore.read(FILE)
print(f"{len(df):,} righe x {len(df.columns)} colonne")
df.head()

## Lo schema, come tabella

Per ogni colonna: tipo, quanti valori distinti, quanti vuoti, e i valori stessi
quando sono pochi. È la cosa da guardare **prima** di scrivere qualunque
aggregazione.

In [ ]:
explore.schema(df)

## L'insidia dei file MCP: la colonna `row`

I file di statistiche del Match Charting Project hanno una riga per ogni
**livello** di aggregazione, più una riga `Total`. Sommarle tutte conta gli
stessi punti più volte.

`row` significa una cosa diversa in ogni file: la lunghezza dello scambio in
`Rally`, il tipo di punto in `KeyPointsServe`, il tipo di colpo in `ShotTypes`.
Guarda sempre quali valori ha prima di filtrare.

In [ ]:
if "row" in df.columns:
    display(df["row"].value_counts())

In [ ]:
# Filtrare al livello giusto: qui la lunghezza dello scambio, escludendo Total
scambi = df[df["row"].isin(["1-3", "4-6", "7-9", "10"])]
print(f"{len(scambi):,} righe dopo il filtro (erano {len(df):,})")
scambi.head()

## Guardare un file di TennisMyLife

Stessa procedura, schema completamente diverso: una riga per match.

In [ ]:
tml = explore.read("tml/atp/2025.csv")
print(f"{len(tml):,} match nel 2025")
explore.schema(tml)

In [ ]:
# Le prime righe, solo le colonne che interessano
tml[["tourney_name", "surface", "round", "winner_name", "loser_name",
     "score", "w_ace", "l_ace", "w_svpt", "l_svpt"]].head(10)